# Experiment 9: Closing the reviewer gaps (alpha sensitivity, RRF baseline, BM25 sanity check, honest oracle)

Runs on the same BEIR datasets as Exp7 and answers the objections a reviewer would raise against Exp7:

1. **Alpha sensitivity** - is the hybrid result an artifact of the fixed alpha=0.75? (sweep 0..1, per-dataset best alpha, leave-one-dataset-out alpha vs fixed)
2. **RRF baseline** - the standard fusion competitor (k = 10, 60, 100) with paired bootstrap vs hybrid alpha=0.75
3. **BM25 sanity check** - is our own sparse BM25 implementation close to published BEIR BM25 numbers? (full test query sets, BM25 only)
4. **Honest oracle / stability** - how much of the ~8-point oracle headroom is real? Queries are perturbed (10% of words dropped); the method is chosen on the *perturbed* runs and scored on the *original* query. If per-query winners were a stable property of the query, this "honest oracle" would beat the best fixed method; if it is noise, it will not.

**Runtime:** works on CPU (MiniLM only, ~30-60 min) or GPU (all 3 models). Every dataset x model result is checkpointed to Google Drive (`AI_Project/RAG Research/Exp9`) - after a disconnect just Run all again.

In [1]:
!pip install -q sentence-transformers beir scikit-learn scipy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 21.5 MB/s eta 0:00:00


## Shared helpers (same as Exp7)

In [2]:
import numpy as np, pandas as pd, re, json, time, os
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor

METHODS = ['dense', 'hybrid', 'rerank']          # ordered cheapest -> most expensive
LAMBDAS = np.r_[0.0, np.geomspace(0.005, 1.0, 14)]

# ---------------------------------------------------------------- BM25 (fast, sparse, vectorised)
class BM25Index:
    def __init__(self, docs, k1=1.2, b=0.75):
        self.cv = CountVectorizer(stop_words='english', lowercase=True)
        tf = self.cv.fit_transform(docs).tocsr().astype(np.float32)
        N = tf.shape[0]
        df = np.asarray((tf > 0).sum(axis=0)).ravel()
        self.idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = max(dl.mean(), 1e-9)
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        data = tf.data * (k1 + 1) / denom * self.idf[tf.col]
        self.M = sparse.csr_matrix((data, (tf.row, tf.col)), shape=tf.shape)
        self.MT = self.M.T.tocsr()

    def score(self, queries, batch=256):
        out = []
        for s in range(0, len(queries), batch):
            Q = self.cv.transform(queries[s:s + batch]).astype(np.float32).tocsr()
            Q.data[:] = 1.0
            out.append((Q @ self.MT).toarray().astype(np.float32))
        return np.vstack(out)

    def query_stats(self, qs):
        an = self.cv.build_analyzer(); vocab = self.cv.vocabulary_
        rows = []
        for q in qs:
            toks = an(q)
            inv = [t for t in toks if t in vocab]
            idfs = [float(self.idf[vocab[t]]) for t in inv]
            rows.append(dict(
                n_tok=len(q.split()), n_terms=len(toks),
                frac_oov=1 - len(inv) / max(1, len(toks)),
                mean_idf=float(np.mean(idfs)) if idfs else 0.0,
                max_idf=max(idfs) if idfs else 0.0,
                has_digit=float(bool(re.search(r'\d', q))),
                has_special=float(bool(re.search(r'[_/\\.\-:=]', q))),
                n_chars=len(q)))
        return pd.DataFrame(rows)

# ---------------------------------------------------------------- array utils
def minmax_rows(S):
    mn = S.min(1, keepdims=True); mx = S.max(1, keepdims=True)
    return (S - mn) / (mx - mn + 1e-9)

def topk(S, k):
    k = min(k, S.shape[1])
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]
    vals = np.take_along_axis(S, part, 1)
    order = np.argsort(-vals, axis=1)
    return np.take_along_axis(part, order, 1)

def topvals(S, k=20):
    k = min(k, S.shape[1])
    return -np.sort(-np.partition(S, S.shape[1] - k, axis=1)[:, -k:], axis=1)

# ---------------------------------------------------------------- metrics (graded relevance supported)
def ndcg_at_k(ranked, rel, k=10):
    dcg = sum(rel.get(i, 0) / np.log2(r + 2) for r, i in enumerate(ranked[:k]))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum(g / np.log2(r + 2) for r, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def eval_lists(tops, rels, k=10):
    nd, rc, mr = [], [], []
    for lst, rel in zip(tops, rels):
        lst = list(lst[:k])
        nd.append(ndcg_at_k(lst, rel, k))
        pos = {i for i, g in rel.items() if g > 0}
        rc.append(len(pos & set(lst)) / max(1, len(pos)))
        r_ = 0.0
        for r, i in enumerate(lst):
            if i in pos:
                r_ = 1.0 / (r + 1); break
        mr.append(r_)
    return np.array(nd), np.array(rc), np.array(mr)

# ---------------------------------------------------------------- retrieval stack (Dense / Hybrid / Hybrid+Rerank)
def run_stack(model_name, doc_texts, q_texts, bm, reranker, exclude_idx=None, alpha=0.75,
              shortlist=20, q_prefix='', device=None, out_k=10):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = 256
    D = model.encode(doc_texts, batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=True)
    Q = model.encode([q_prefix + q for q in q_texts], batch_size=128, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    S_d = (Q @ D.T).astype(np.float32)
    S_b = bm.score(q_texts)
    if exclude_idx is not None:                       # ignore identical ids (e.g. ArguAna)
        for i, j in enumerate(exclude_idx):
            if j is not None:
                S_d[i, j] = S_d[i].min(); S_b[i, j] = S_b[i].min()
    H = alpha * minmax_rows(S_d) + (1 - alpha) * minmax_rows(S_b)
    top_d = topk(S_d, out_k)
    top_h_full = topk(H, shortlist)
    pairs = [(q_texts[i], doc_texts[j]) for i in range(len(q_texts)) for j in top_h_full[i]]
    sc = np.asarray(reranker.predict(pairs, batch_size=64, show_progress_bar=True)).reshape(len(q_texts), -1)
    order = np.argsort(-sc, axis=1)
    top_r = np.take_along_axis(top_h_full, order, 1)[:, :out_k]
    return dict(S_d=S_d, S_b=S_b, H=H, top_d=top_d, top_h=top_h_full[:, :out_k], top_r=top_r,
                model=model, D=D)

def measure_latency(model, D, bm, reranker, doc_texts, q_sample, alpha=0.75, shortlist=20, q_prefix=''):
    """Per-query wall-clock latency in ms for [dense, hybrid, rerank] (each includes the previous stage)."""
    for q in q_sample[:3]:
        model.encode([q_prefix + q], normalize_embeddings=True, show_progress_bar=False)
    td, th, tr = [], [], []
    for q in q_sample:
        t0 = time.perf_counter()
        qe = model.encode([q_prefix + q], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        sd = (qe @ D.T).astype(np.float32)
        _ = topk(sd, 10)
        t1 = time.perf_counter()
        sb = bm.score([q])
        h = alpha * minmax_rows(sd) + (1 - alpha) * minmax_rows(sb)
        sh = topk(h, shortlist)[0]
        t2 = time.perf_counter()
        _ = reranker.predict([(q, doc_texts[j]) for j in sh], show_progress_bar=False)
        t3 = time.perf_counter()
        td.append(t1 - t0); th.append(t2 - t0); tr.append(t3 - t0)
    return [1000 * float(np.mean(td)), 1000 * float(np.mean(th)), 1000 * float(np.mean(tr))]

# ---------------------------------------------------------------- label-free routing features
def build_features(S_d, S_b, H, q_texts, bm):
    d, b, h = topvals(S_d), topvals(S_b), topvals(H)
    kk = d.shape[1]
    f = pd.DataFrame({
        'd_top1': d[:, 0], 'd_margin': d[:, 0] - d[:, 1], 'd_spread': d[:, 0] - d[:, kk - 1], 'd_std': d.std(1),
        'b_top1': b[:, 0], 'b_margin': (b[:, 0] - b[:, 1]) / (b[:, 0] + 1e-6),
        'b_spread': (b[:, 0] - b[:, kk - 1]) / (b[:, 0] + 1e-6),
        'h_margin': h[:, 0] - h[:, 1], 'h_spread': h[:, 0] - h[:, kk - 1]})
    td, tb = topk(S_d, 10), topk(S_b, 10)
    f['overlap10'] = [len(set(a) & set(c)) / 10.0 for a, c in zip(td, tb)]
    f['top1_agree'] = (td[:, 0] == tb[:, 0]).astype(float)
    qs = bm.query_stats(q_texts)
    return pd.concat([qs.reset_index(drop=True), f.reset_index(drop=True)], axis=1)

# ---------------------------------------------------------------- routing analysis
def router_cost(choice, t):
    t = np.asarray(t, float)
    base = np.full(len(choice), t[1])                 # features need dense + BM25 scores
    base[choice == 2] = t[2]
    return float(base.mean())

def bootstrap_diff(a, b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed); nq = len(a)
    idx = rng.integers(0, nq, size=(n_boot, nq))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return dict(diff=float(a.mean() - b.mean()), ci_lo=float(lo), ci_hi=float(hi), p_le0=float((d <= 0).mean()))

def fit_gain_models(Xtr, Ytr, seed=42):
    models = {}
    for m in (1, 2):
        reg = GradientBoostingRegressor(n_estimators=150, max_depth=2, learning_rate=0.05,
                                        subsample=0.8, random_state=seed)
        reg.fit(Xtr, Ytr[:, m] - Ytr[:, 0]); models[m] = reg
    return models

def predict_gains(models, Xte):
    P = np.zeros((len(Xte), 3))
    for m, reg in models.items():
        P[:, m] = reg.predict(Xte)
    return P

def choose(P, lam, t):
    cn = np.asarray(t, float) / t[2]
    U = P - lam * (cn - cn[0])[None, :]
    U[:, 0] = 0.0
    return U.argmax(1)

def _row(nd, rc, mr, cost):
    return dict(ndcg=float(nd.mean()), recall=float(rc.mean()), mrr=float(mr.mean()), cost=float(cost))

def analyze_routing(ND, RC, MR, t, X, seed=42, n_boot=5000, n_splits=5):
    nq = len(ND); ar = np.arange(nq); t = np.asarray(t, float)
    Xv = X.values.astype(float)
    res = {'n_queries': nq, 't': t.tolist()}
    best_fixed = int(ND.mean(0).argmax())
    tab = {}
    for i, name in enumerate(['Dense', 'Hybrid', 'Rerank']):
        tab[name] = _row(ND[:, i], RC[:, i], MR[:, i], t[i])
    # rule router (label-free, query text only)
    ntok = X['n_tok'].values; has_d = X['has_digit'].values > 0
    rc_ = np.where(has_d & (ntok < 12), 0, np.where(ntok >= 14, 1, 2))
    tab['Rule router'] = _row(ND[ar, rc_], RC[ar, rc_], MR[ar, rc_], t[rc_].mean())
    # oracle (ties -> cheapest because columns are ordered by cost)
    oc = ND.argmax(1)
    tab['Oracle'] = _row(ND[ar, oc], RC[ar, oc], MR[ar, oc], t[oc].mean())
    res['oracle_label_counts'] = {METHODS[i]: int((oc == i).sum()) for i in range(3)}
    res['n_informative'] = int((ND.max(1) - ND.min(1) > 1e-9).sum())
    # learned router: out-of-fold predicted gain over dense
    P = np.zeros((nq, 3))
    kf = KFold(n_splits=min(n_splits, nq), shuffle=True, random_state=seed)
    for tr, te in kf.split(Xv):
        P[te] = predict_gains(fit_gain_models(Xv[tr], ND[tr], seed), Xv[te])
    sweep = []
    for lam in LAMBDAS:
        ch = choose(P, lam, t)
        r = _row(ND[ar, ch], RC[ar, ch], MR[ar, ch], router_cost(ch, t))
        r.update(lam=float(lam), frac_rerank=float((ch == 2).mean()), frac_dense=float((ch == 0).mean()))
        sweep.append(r)
    ch0 = choose(P, 0.0, t)
    tab['Learned router'] = _row(ND[ar, ch0], RC[ar, ch0], MR[ar, ch0], router_cost(ch0, t))
    res['learned_sweep'] = sweep
    res['best_fixed'] = ['Dense', 'Hybrid', 'Rerank'][best_fixed]
    res['boot_learned_vs_best_fixed'] = bootstrap_diff(ND[ar, ch0], ND[:, best_fixed], n_boot, seed)
    bf_nd = ND[:, best_fixed].mean()
    ok = [s['cost'] for s in sweep if s['ndcg'] >= bf_nd]
    res['min_cost_to_match_best_fixed'] = float(min(ok)) if ok else None
    res['oracle_gap_closed'] = float((tab['Learned router']['ndcg'] - bf_nd) /
                                     max(tab['Oracle']['ndcg'] - bf_nd, 1e-9))
    # confidence-threshold (score-margin) baselines, swept as Pareto curves
    conf = X['d_margin'].values; order = np.argsort(conf)
    curves = {}
    for name, base in [('Margin router (dense/rerank)', 0), ('Margin router (hybrid/rerank)', 1)]:
        pts = []
        for q in np.linspace(0, 1, 21):
            k = int(round(q * nq)); ch = np.full(nq, base); ch[order[:k]] = 2
            pts.append(dict(q=float(q), ndcg=float(ND[ar, ch].mean()), cost=float(t[ch].mean())))
        curves[name] = pts
    res['margin_curves'] = curves
    res['table'] = tab
    res['_P'] = P
    return res

def lodo_routing(sets, n_boot=5000, seed=42):
    """Leave-one-set-out: train the gain regressors on all other sets, test on the held-out one."""
    out = {}
    names = list(sets.keys())
    if len(names) < 2:
        return out
    for held in names:
        tr_names = [k for k in names if k != held]
        Xtr = np.vstack([sets[k]['X'].values.astype(float) for k in tr_names])
        Ytr = np.vstack([sets[k]['ND'] for k in tr_names])
        models = fit_gain_models(Xtr, Ytr, seed)
        S = sets[held]; ND = S['ND']; ar = np.arange(len(ND)); t = np.asarray(S['t'], float)
        P = predict_gains(models, S['X'].values.astype(float))
        sweep = []
        for lam in LAMBDAS:
            ch = choose(P, lam, t)
            sweep.append(dict(lam=float(lam), ndcg=float(ND[ar, ch].mean()), cost=router_cost(ch, t)))
        bf = int(ND.mean(0).argmax()); ch0 = choose(P, 0.0, t)
        out[held] = dict(best_fixed=['Dense', 'Hybrid', 'Rerank'][bf], best_fixed_ndcg=float(ND[:, bf].mean()),
                         router_ndcg=float(ND[ar, ch0].mean()), sweep=sweep,
                         boot=bootstrap_diff(ND[ar, ch0], ND[:, bf], n_boot, seed))
    return out

# ---------------------------------------------------------------- reporting (tables, LaTeX, figures)
ROW_ORDER = ['Dense', 'Hybrid', 'Rerank', 'Rule router', 'Learned router', 'Oracle']

def main_table(RES_model, metric='ndcg'):
    dsets = list(RES_model.keys())
    df = pd.DataFrame({d: {r: RES_model[d]['table'][r][metric] for r in ROW_ORDER} for d in dsets})
    df['Avg'] = df.mean(axis=1)
    if metric != 'cost':
        df['Latency (ms)'] = pd.Series({r: np.mean([RES_model[d]['table'][r]['cost'] for d in dsets]) for r in ROW_ORDER})
    return df.loc[ROW_ORDER]

def _esc(s):
    return str(s).replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')

def df_to_latex(df, caption, label, fmt='{:.3f}', lat_fmt='{:.1f}'):
    cols = list(df.columns)
    best = {c: df.loc[[r for r in df.index if r != 'Oracle'], c].idxmax() for c in cols if c != 'Latency (ms)'}
    lines = ['\\begin{table}[t]', '\\centering', '\\caption{' + _esc(caption) + '}', '\\label{' + label + '}',
             '\\begin{tabular}{l' + 'c' * len(cols) + '}', '\\toprule',
             ' & ' + ' & '.join(_esc(c) for c in cols) + ' \\\\', '\\midrule']
    for r in df.index:
        cells = []
        for c in cols:
            v = df.loc[r, c]
            s = lat_fmt.format(v) if c == 'Latency (ms)' else fmt.format(v)
            if c != 'Latency (ms)' and best.get(c) == r:
                s = '\\textbf{' + s + '}'
            cells.append(s)
        lines.append(_esc(r) + ' & ' + ' & '.join(cells) + ' \\\\')
        if r == 'Rerank':
            lines.append('\\midrule')
    lines += ['\\bottomrule', '\\end{tabular}', '\\end{table}']
    return '\n'.join(lines)

def significance_table(RES):
    rows = []
    for m, dd in RES.items():
        for d, r in dd.items():
            b = r['boot_learned_vs_best_fixed']
            rows.append(dict(model=m, dataset=d, best_fixed=r['best_fixed'],
                             best_fixed_ndcg=r['table'][r['best_fixed']]['ndcg'],
                             learned_ndcg=r['table']['Learned router']['ndcg'], diff=b['diff'],
                             ci_lo=b['ci_lo'], ci_hi=b['ci_hi'], significant=bool(b['ci_lo'] > 0),
                             oracle_ndcg=r['table']['Oracle']['ndcg'], oracle_gap_closed=r['oracle_gap_closed'],
                             min_cost_to_match=r['min_cost_to_match_best_fixed']))
    return pd.DataFrame(rows)

def lodo_table(LODO):
    rows = []
    for m, dd in LODO.items():
        for d, r in dd.items():
            rows.append(dict(model=m, held_out=d, best_fixed=r['best_fixed'], best_fixed_ndcg=r['best_fixed_ndcg'],
                             router_ndcg=r['router_ndcg'], diff=r['boot']['diff'], ci_lo=r['boot']['ci_lo'],
                             ci_hi=r['boot']['ci_hi'], significant=bool(r['boot']['ci_lo'] > 0)))
    return pd.DataFrame(rows)

def plot_pareto(RES, path_prefix):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    models = list(RES.keys()); dsets = list(RES[models[0]].keys())
    fig, axes = plt.subplots(len(models), len(dsets), figsize=(3.3 * len(dsets), 2.7 * len(models)), squeeze=False)
    for i, m in enumerate(models):
        for j, d in enumerate(dsets):
            ax = axes[i][j]; r = RES[m][d]
            sw = r['learned_sweep']
            ax.plot([s['cost'] for s in sw], [s['ndcg'] for s in sw], '-o', ms=3, lw=1.2, label='Learned router')
            for name, pts in r['margin_curves'].items():
                ax.plot([p['cost'] for p in pts], [p['ndcg'] for p in pts], '--', lw=1, label=name)
            for name, mk in [('Dense', 's'), ('Hybrid', '^'), ('Rerank', 'D')]:
                ax.scatter(r['table'][name]['cost'], r['table'][name]['ndcg'], marker=mk, s=28, zorder=5, label=name)
            ax.scatter(r['table']['Oracle']['cost'], r['table']['Oracle']['ndcg'], marker='*', s=70, c='k', zorder=6, label='Oracle')
            ax.set_xscale('log')
            if i == 0: ax.set_title(d, fontsize=9)
            if j == 0: ax.set_ylabel(m + '\nnDCG@10', fontsize=8)
            if i == len(models) - 1: ax.set_xlabel('Latency (ms/query)', fontsize=8)
            ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=4, fontsize=7, frameon=False)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.savefig(path_prefix + '.pdf'); fig.savefig(path_prefix + '.png', dpi=300)
    plt.close(fig)

def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items() if not str(k).startswith('_')}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


## Exp9 helpers

In [3]:

from scipy.stats import spearmanr

ALPHAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0]   # 0 = BM25 only, 1 = dense only
RRF_KS = [10, 60, 100]

def alpha_sweep(S_d, S_b, rels, alphas=ALPHAS):
    rows, perq = [], {}
    nd_, nb_ = minmax_rows(S_d), minmax_rows(S_b)
    for a in alphas:
        nd, rc, mr = eval_lists(topk(a * nd_ + (1 - a) * nb_, 10), rels)
        rows.append(dict(alpha=a, ndcg=float(nd.mean()), recall=float(rc.mean()), mrr=float(mr.mean())))
        perq[a] = nd
    return rows, perq

def rrf_top10(S_d, S_b, k=60, depth=100):
    td, tb = topk(S_d, depth), topk(S_b, depth); out = []
    for a, b in zip(td, tb):
        sc = {}
        for r, i in enumerate(a): sc[i] = sc.get(i, 0.0) + 1.0 / (k + r + 1)
        for r, i in enumerate(b): sc[i] = sc.get(i, 0.0) + 1.0 / (k + r + 1)
        out.append([i for i, _ in sorted(sc.items(), key=lambda x: -x[1])[:10]])
    return out

def perturb_query(q, rate, rng):
    w = q.split()
    if len(w) < 4: return None
    n_del = max(1, int(round(rate * len(w))))
    drop = set(rng.sample(range(len(w)), n_del))
    return ' '.join(x for i, x in enumerate(w) if i not in drop)

def _rho(a, b):
    r = spearmanr(a, b).correlation
    return float(r) if r == r else None

def analyze_combo(S_d0, S_b0, rels0, tops, rels_all, nq, cand, K, seed=42):
    """S_*0: score matrices of the ORIGINAL queries. tops: (top_d, top_h, top_r) over originals + perturbed queries."""
    out = {}
    rows, perq = alpha_sweep(S_d0, S_b0, rels0)
    out['alpha_rows'] = rows
    rrf_rows = []
    for k in RRF_KS:
        nd, rc, mr = eval_lists(rrf_top10(S_d0, S_b0, k=k), rels0)
        rrf_rows.append(dict(k=k, ndcg=float(nd.mean()), recall=float(rc.mean()), mrr=float(mr.mean())))
        if k == 60:
            out['rrf_vs_hybrid075'] = bootstrap_diff(nd, perq[0.75], n_boot=3000, seed=seed)
            out['rrf_vs_best_fixed_alpha'] = None
    out['rrf_rows'] = rrf_rows
    # ---- honest oracle + stability under query perturbation
    ev = [eval_lists(t, rels_all) for t in tops]
    ND_all = np.stack([e[0] for e in ev], 1)
    pi = len(cand)
    out['n_perturbable'] = pi
    if pi >= 20 and K >= 1:
        O = ND_all[:nq][cand]
        P = ND_all[nq:].reshape(K, pi, 3)
        Pm = P.mean(0); ar = np.arange(pi)
        ch = Pm.argmax(1)                       # method chosen using INDEPENDENT perturbed runs
        bf = int(O.mean(0).argmax())
        oracle = float(O.max(1).mean()); honest = float(O[ar, ch].mean()); best_fixed = float(O[:, bf].mean())
        ow = O.argmax(1)
        spread = O.max(1) - np.sort(O, 1)[:, -2]
        clear = spread >= 0.05
        p = np.bincount(ow[clear], minlength=3) / max(1, clear.sum())
        out['oracle_honesty'] = dict(
            n=pi, best_fixed=['Dense', 'Hybrid', 'Rerank'][bf], best_fixed_ndcg=best_fixed,
            oracle_ndcg=oracle, honest_oracle_ndcg=honest,
            oracle_headroom=oracle - best_fixed, exploitable_headroom=honest - best_fixed,
            boot_honest_vs_best_fixed=bootstrap_diff(O[ar, ch], O[:, bf], n_boot=3000, seed=seed),
            winner_agreement=(float((ow[clear] == ch[clear]).mean()) if clear.any() else None),
            winner_agreement_chance=float((p ** 2).sum()), n_clear_winner=int(clear.sum()),
            rho_difficulty_dense=_rho(O[:, 0], Pm[:, 0]), rho_difficulty_hybrid=_rho(O[:, 1], Pm[:, 1]),
            rho_difficulty_rerank=_rho(O[:, 2], Pm[:, 2]),
            rho_gain_hybrid_vs_dense=_rho(O[:, 1] - O[:, 0], Pm[:, 1] - Pm[:, 0]),
            rho_gain_rerank_vs_dense=_rho(O[:, 2] - O[:, 0], Pm[:, 2] - Pm[:, 0]),
            rho_gain_rerank_vs_hybrid=_rho(O[:, 2] - O[:, 1], Pm[:, 2] - Pm[:, 1]))
    return out

def bm25_only_eval(bm, qs, rels, exclude, batch=128):
    tops = []
    for s in range(0, len(qs), batch):
        S = bm.score(qs[s:s + batch])
        for i in range(S.shape[0]):
            j = exclude[s + i]
            if j is not None: S[i, j] = S[i].min()
        tops.append(topk(S, 10))
    nd, rc, mr = eval_lists(np.vstack(tops), rels)
    return float(nd.mean()), float(rc.mean())

def lodo_alpha(R, model, datasets):
    ds = [d for d in datasets if (d, model) in R]
    tab = {d: {r['alpha']: r['ndcg'] for r in R[(d, model)]['alpha_rows']} for d in ds}
    rows = []
    for held in ds:
        others = [d for d in ds if d != held]
        a_l = max(ALPHAS, key=lambda a: np.mean([tab[d][a] for d in others])) if others else 0.75
        a_b = max(tab[held], key=tab[held].get)
        rows.append(dict(model=model, dataset=held, alpha_lodo=a_l, ndcg_lodo=tab[held][a_l], ndcg_fixed_075=tab[held][0.75],
                         alpha_best=a_b, ndcg_best_alpha=tab[held][a_b], ndcg_dense=tab[held][1.0], ndcg_bm25=tab[held][0.0]))
    return pd.DataFrame(rows)

def simple_latex(df, caption, label, fmt='{:.3f}'):
    cols = list(df.columns)
    esc = lambda s: str(s).replace('_', '\\_').replace('%', '\\%').replace('&', '\\&')
    lines = ['\\begin{table}[t]', '\\centering', '\\caption{' + esc(caption) + '}', '\\label{' + label + '}',
             '\\begin{tabular}{' + 'l' * 2 + 'c' * (len(cols) - 2) + '}', '\\toprule',
             ' & '.join(esc(c) for c in cols) + ' \\\\', '\\midrule']
    for _, r in df.iterrows():
        cells = [fmt.format(v) if isinstance(v, (float, np.floating)) else esc(v) for v in r.values]
        lines.append(' & '.join(cells) + ' \\\\')
    lines += ['\\bottomrule', '\\end{tabular}', '\\end{table}']
    return '\n'.join(lines)


## Config

In [4]:
import torch, random, gc
from sentence_transformers import CrossEncoder
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/AI_Project/RAG Research/Exp9'
os.makedirs(OUT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
USE_FIQA = False        # FiQA (57k docs) is only practical on GPU; set True there (BM25 sanity always includes it)
DATASETS = ['scifact', 'nfcorpus', 'arguana'] + (['fiqa'] if USE_FIQA else [])
ALL_MODELS = {'MiniLM-L6': 'sentence-transformers/all-MiniLM-L6-v2',
              'mpnet-base': 'sentence-transformers/all-mpnet-base-v2',
              'bge-small': 'BAAI/bge-small-en-v1.5'}
MODELS = ALL_MODELS if DEVICE == 'cuda' else {'MiniLM-L6': ALL_MODELS['MiniLM-L6']}
QUERY_PREFIX = {'bge-small': 'Represent this sentence for searching relevant passages: '}
ALPHA, SHORTLIST = 0.75, 20
MAX_QUERIES = 500       # queries per dataset for the alpha / RRF analysis
PERTURB_N = 250         # first N queries (with >=4 words) also get perturbed copies
K_PERTURB = 3           # perturbed copies per query
PERTURB_RATE = 0.10     # fraction of words dropped
CONFIG = dict(datasets=DATASETS, models=MODELS, alpha=ALPHA, shortlist=SHORTLIST, max_queries=MAX_QUERIES,
              perturb_n=PERTURB_N, k_perturb=K_PERTURB, perturb_rate=PERTURB_RATE, device=DEVICE,
              gpu=(torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'))
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=256, device=DEVICE)

def load_beir(name, max_queries=1000, seed=42):
    from beir import util
    from beir.datasets.data_loader import GenericDataLoader
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip"
    path = util.download_and_unzip(url, '/content/beir')
    corpus, queries, qrels = GenericDataLoader(data_folder=path).load(split='test')
    doc_ids = list(corpus.keys()); id2idx = {d: i for i, d in enumerate(doc_ids)}
    docs = [(corpus[d].get('title', '') + ' ' + corpus[d].get('text', '')).strip() for d in doc_ids]
    qids = sorted(q for q in queries if q in qrels and any(s > 0 and d in id2idx for d, s in qrels[q].items()))
    if max_queries and len(qids) > max_queries:
        qids = sorted(random.Random(seed).sample(qids, max_queries))
    rels = [{id2idx[d]: s for d, s in qrels[q].items() if s > 0 and d in id2idx} for q in qids]
    return dict(docs=docs, qs=[queries[q] for q in qids], rels=rels, exclude=[id2idx.get(q) for q in qids])

Mounted at /content/drive
device: cuda


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Part A: BM25 sanity check (full test query sets, BM25 only, no embeddings)
Compare against the published BEIR BM25 nDCG@10. The reference values below are the ones I remember from the BEIR paper - **verify them against the paper (Table of BM25 results) before citing**. Small differences are expected (tokenizer, stop-words, BM25 parameters; ArguAna in particular differs a lot between BM25 implementations).

In [5]:
REF_BM25 = {'scifact': 0.665, 'nfcorpus': 0.325, 'arguana': 0.315, 'fiqa': 0.236}   # verify against the BEIR paper!
SAN_FILE = f'{OUT}/bm25_sanity.csv'
if os.path.exists(SAN_FILE):
    san = pd.read_csv(SAN_FILE); print('loaded from Drive')
else:
    rows = []
    for ds in ['scifact', 'nfcorpus', 'arguana', 'fiqa']:
        d = load_beir(ds, max_queries=None)
        bm_ = BM25Index(d['docs'])
        nd, rc = bm25_only_eval(bm_, d['qs'], d['rels'], d['exclude'])
        rows.append(dict(dataset=ds, n_docs=len(d['docs']), n_queries=len(d['qs']), our_bm25_ndcg10=nd,
                         reference_ndcg10=REF_BM25[ds], diff=nd - REF_BM25[ds]))
        print(rows[-1]); del bm_, d; gc.collect()
    san = pd.DataFrame(rows); san.to_csv(SAN_FILE, index=False)
print(san.round(3).to_string())

/content/beir/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

{'dataset': 'scifact', 'n_docs': 5183, 'n_queries': 300, 'our_bm25_ndcg10': 0.6639464256283998, 'reference_ndcg10': 0.665, 'diff': -0.0010535743716002521}


/content/beir/nfcorpus.zip:   0%|          | 0.00/2.34M [00:00<?, ?iB/s]

  0%|          | 0/3633 [00:00<?, ?it/s]

...

  0%|          | 0/8674 [00:00<?, ?it/s]

{'dataset': 'arguana', 'n_docs': 8674, 'n_queries': 1401, 'our_bm25_ndcg10': 0.44372600337640067, 'reference_ndcg10': 0.315, 'diff': 0.12872600337640067}


/content/beir/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

{'dataset': 'fiqa', 'n_docs': 57638, 'n_queries': 648, 'our_bm25_ndcg10': 0.2256046037209778, 'reference_ndcg10': 0.236, 'diff': -0.010395396279022195}
    dataset  n_docs  n_queries  our_bm25_ndcg10  reference_ndcg10   diff
0   scifact    5183        300            0.664             0.665 -0.001
1  nfcorpus    3633        323            0.310             0.325 -0.015
2   arguana    8674       1401            0.444             0.315  0.129
3      fiqa   57638        648            0.226             0.236 -0.010


## Part B: Per dataset x model - alpha sweep, RRF, honest oracle (checkpointed to Drive)

In [6]:
R = {}
for ds in DATASETS:
    data, bm = None, None
    for mname, mpath in MODELS.items():
        ck = f'{OUT}/ckpt_{ds}_{mname}.json'
        if os.path.exists(ck):
            R[(ds, mname)] = json.load(open(ck)); print(f'{ds}/{mname}: loaded checkpoint'); continue
        if data is None:
            data = load_beir(ds, MAX_QUERIES); bm = BM25Index(data['docs'])
            print(f"\n##### {ds}: {len(data['docs'])} docs, {len(data['qs'])} queries")
        print(f'--- {ds} / {mname}')
        nq = len(data['qs']); rng = random.Random(42)
        cand = [i for i in range(min(nq, PERTURB_N)) if len(data['qs'][i].split()) >= 4]
        pert = [[perturb_query(data['qs'][i], PERTURB_RATE, rng) for i in cand] for _ in range(K_PERTURB)]
        q_all = data['qs'] + [t for lst in pert for t in lst]
        rels_all = data['rels'] + [data['rels'][i] for _ in range(K_PERTURB) for i in cand]
        excl_all = data['exclude'] + [data['exclude'][i] for _ in range(K_PERTURB) for i in cand]
        pref = QUERY_PREFIX.get(mname, '')
        st = run_stack(mpath, data['docs'], q_all, bm, reranker, exclude_idx=excl_all, alpha=ALPHA,
                       shortlist=SHORTLIST, q_prefix=pref, device=DEVICE)
        res = analyze_combo(st['S_d'][:nq], st['S_b'][:nq], data['rels'], (st['top_d'], st['top_h'], st['top_r']),
                            rels_all, nq, cand, K_PERTURB)
        R[(ds, mname)] = res
        json.dump(to_jsonable(res), open(ck, 'w'))                     # checkpoint after every combo
        a = {r['alpha']: r['ndcg'] for r in res['alpha_rows']}
        print(f"   nDCG@10  bm25 {a[0.0]:.3f} | dense {a[1.0]:.3f} | hybrid(.75) {a[0.75]:.3f} | best-alpha {max(a.values()):.3f} "
              f"(alpha={max(a, key=a.get)}) | RRF(60) {[r['ndcg'] for r in res['rrf_rows'] if r['k'] == 60][0]:.3f}")
        if 'oracle_honesty' in res:
            h = res['oracle_honesty']
            print(f"   oracle {h['oracle_ndcg']:.3f} | honest oracle {h['honest_oracle_ndcg']:.3f} | best fixed {h['best_fixed_ndcg']:.3f}"
                  f" | rho(gain) hybrid-dense {h['rho_gain_hybrid_vs_dense']}")
        del st; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

  0%|          | 0/5183 [00:00<?, ?it/s]


##### scifact: 5183 docs, 300 queries
--- scifact / MiniLM-L6


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

   nDCG@10  bm25 0.425 | dense 0.489 | hybrid(.75) 0.530 | best-alpha 0.534 (alpha=0.8) | RRF(60) 0.515
   oracle 0.672 | honest oracle 0.657 | best fixed 0.512 | rho(gain) hybrid-dense 0.9030437284814917
--- arguana / bge-small


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/68 [00:00<?, ?it/s]

Batches:   0%|          | 0/391 [00:00<?, ?it/s]

   nDCG@10  bm25 0.425 | dense 0.606 | hybrid(.75) 0.581 | best-alpha 0.610 (alpha=0.9) | RRF(60) 0.544
   oracle 0.717 | honest oracle 0.708 | best fixed 0.598 | rho(gain) hybrid-dense 0.8707955139894948


## Part C: Tables, figures, LaTeX (saved to Drive)

In [7]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
models = list(MODELS.keys())

# ---- alpha sweep csv + figure + leave-one-dataset-out alpha
rows = [dict(dataset=d, model=m, **r) for (d, m), v in R.items() for r in v['alpha_rows']]
pd.DataFrame(rows).to_csv(f'{OUT}/alpha_sweep.csv', index=False)
fig, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 2.8), squeeze=False)
for j, m in enumerate(models):
    ax = axes[0][j]
    for d in DATASETS:
        if (d, m) in R:
            ax.plot([r['alpha'] for r in R[(d, m)]['alpha_rows']], [r['ndcg'] for r in R[(d, m)]['alpha_rows']], '-o', ms=3, label=d)
    ax.axvline(0.75, ls='--', c='gray', lw=0.8); ax.set_title(m, fontsize=9); ax.set_xlabel('alpha (0=BM25, 1=dense)', fontsize=8)
    if j == 0: ax.set_ylabel('nDCG@10', fontsize=8)
    ax.tick_params(labelsize=7); ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(f'{OUT}/alpha_sweep.pdf'); fig.savefig(f'{OUT}/alpha_sweep.png', dpi=300)

lod = pd.concat([lodo_alpha(R, m, DATASETS) for m in models]); lod.to_csv(f'{OUT}/alpha_lodo.csv', index=False)
print('===== Alpha: fixed 0.75 vs leave-one-dataset-out alpha vs best alpha (oracle alpha) =====')
print(lod.round(3).to_string())

# ---- RRF vs hybrid table
rr_rows = []
for (d, m), v in R.items():
    a = {r['alpha']: r['ndcg'] for r in v['alpha_rows']}
    rrf = {r['k']: r['ndcg'] for r in v['rrf_rows']}; b = v['rrf_vs_hybrid075']
    rr_rows.append(dict(dataset=d, model=m, bm25=a[0.0], dense=a[1.0], hybrid_075=a[0.75], best_alpha=max(a.values()),
                        rrf_k10=rrf[10], rrf_k60=rrf[60], rrf_k100=rrf[100], rrf60_minus_hybrid=b['diff'],
                        ci_lo=b['ci_lo'], ci_hi=b['ci_hi']))
rrf_df = pd.DataFrame(rr_rows); rrf_df.to_csv(f'{OUT}/rrf_vs_hybrid.csv', index=False)
open(f'{OUT}/rrf_vs_hybrid.tex', 'w').write(simple_latex(rrf_df.drop(columns=['ci_lo', 'ci_hi']), 'nDCG@10: BM25, dense, hybrid and RRF fusion', 'tab:rrf'))
print('\n===== RRF vs hybrid (nDCG@10; diff CI = paired bootstrap, RRF(k=60) - hybrid(0.75)) =====')
print(rrf_df.round(3).to_string())

# ---- honest oracle / stability
h_rows = [dict(dataset=d, model=m, **v['oracle_honesty']) for (d, m), v in R.items() if 'oracle_honesty' in v]
for r in h_rows:
    b = r.pop('boot_honest_vs_best_fixed'); r['honest_minus_best_ci_lo'] = b['ci_lo']; r['honest_minus_best_ci_hi'] = b['ci_hi']
hon = pd.DataFrame(h_rows); hon.to_csv(f'{OUT}/oracle_honesty.csv', index=False)
print('\n===== Honest oracle & stability (rho = Spearman between original and perturbed-query runs) =====')
print(hon.round(3).to_string())

with open(f'{OUT}/all_results_exp9.json', 'w') as f:
    json.dump(to_jsonable({'config': CONFIG, 'bm25_sanity': san.to_dict('records'),
                           'results': {f'{d}|{m}': v for (d, m), v in R.items()}}), f, indent=1)
print('\nAll outputs saved to', OUT); print(sorted(os.listdir(OUT)))

===== Alpha: fixed 0.75 vs leave-one-dataset-out alpha vs best alpha (oracle alpha) =====
        model   dataset  alpha_lodo  ndcg_lodo  ndcg_fixed_075  alpha_best  ndcg_best_alpha  ndcg_dense  ndcg_bm25
0   MiniLM-L6   scifact        0.80      0.704           0.714        0.60            0.717       0.645      0.664
1   MiniLM-L6  nfcorpus        0.75      0.348           0.348        0.70            0.352       0.316      0.310
2   MiniLM-L6   arguana        0.70      0.524           0.530        0.80            0.539       0.510      0.425
0  mpnet-base   scifact        0.80      0.714           0.724        0.60            0.727       0.641      0.664
1  mpnet-base  nfcorpus        0.75      0.363           0.363        0.70            0.364       0.335      0.310
2  mpnet-base   arguana        0.60      0.514           0.530        0.80            0.534       0.489      0.425
0   bge-small   scifact        0.90      0.713           0.728        0.75            0.728       0.701  

## How to read / write this up
- **Alpha:** if best alpha varies widely across datasets but the leave-one-dataset-out alpha loses little vs fixed 0.75, say the fixed choice is a reasonable default; otherwise report hybrid as dataset-dependent.
- **RRF:** report it as the standard-fusion baseline; claim a difference only where the bootstrap CI excludes 0.
- **BM25 sanity:** if our numbers are within ~1-2 points of the published ones (ArguAna may differ more), state that the custom BM25 is validated; if not, rerun with Pyserini/BM25s and say so.
- **Honest oracle:** `exploitable_headroom` (honest oracle minus best fixed) is the realistic routing headroom; compare it with the raw `oracle_headroom`. Low `rho_gain_*` with high `rho_difficulty_*` means "which queries are hard is stable, but which *method* wins on a query is mostly noise" - this explains why Exp7's router could not learn it.
- Note: with only K=3 perturbed copies the selector itself is noisy, so `exploitable_headroom` is a conservative (lower-bound) estimate - say so.